# Probability & Statistics Interactive Notebook

Essential probability and statistics for machine learning.

**Topics:**
- Probability distributions
- Central Limit Theorem
- Bayes Theorem
- Hypothesis Testing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Probability Distributions

Distributions describe how data is spread. Key distributions in ML:
- **Normal (Gaussian):** Weights initialization, noise modeling
- **Bernoulli/Binomial:** Binary classification
- **Poisson:** Count data, rare events

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Normal distribution
x = np.linspace(-4, 4, 200)
for mu, sigma in [(0,1), (0,0.5), (1,1.5)]:
    axes[0,0].plot(x, stats.norm.pdf(x, mu, sigma), label=f"μ={mu}, σ={sigma}")
axes[0,0].set_title("Normal Distribution"); axes[0,0].legend()

# Binomial
for n, p in [(20,0.5), (20,0.7), (40,0.5)]:
    k = np.arange(0, n+1)
    axes[0,1].bar(k, stats.binom.pmf(k, n, p), alpha=0.5, label=f"n={n}, p={p}")
axes[0,1].set_title("Binomial Distribution"); axes[0,1].legend()

# Poisson
for lam in [1, 4, 10]:
    k = np.arange(0, 20)
    axes[1,0].bar(k, stats.poisson.pmf(k, lam), alpha=0.5, label=f"λ={lam}")
axes[1,0].set_title("Poisson Distribution"); axes[1,0].legend()

# Exponential
x = np.linspace(0, 5, 200)
for lam in [0.5, 1, 2]:
    axes[1,1].plot(x, stats.expon.pdf(x, scale=1/lam), label=f"λ={lam}")
axes[1,1].set_title("Exponential Distribution"); axes[1,1].legend()

plt.tight_layout()
plt.show()

### The Normal Distribution in Detail

The **68-95-99.7 rule**: ~68% of data falls within 1σ, ~95% within 2σ, ~99.7% within 3σ.

In [ ]:
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, y, "k-", linewidth=2)

for n_sigma, alpha in [(1, 0.3), (2, 0.2), (3, 0.1)]:
    mask = (x >= -n_sigma) & (x <= n_sigma)
    pct = (stats.norm.cdf(n_sigma) - stats.norm.cdf(-n_sigma)) * 100
    ax.fill_between(x[mask], y[mask], alpha=alpha+0.2, label=f"{n_sigma}σ: {pct:.1f}%")

ax.set_title("Normal Distribution - 68-95-99.7 Rule")
ax.legend(fontsize=12)
plt.show()

## 2. Central Limit Theorem (CLT)

The CLT states: regardless of the original distribution, the **sampling distribution of the mean** approaches a normal distribution as sample size increases.

This is why the normal distribution appears everywhere in statistics!

In [ ]:
# CLT demonstration with different source distributions
np.random.seed(42)

distributions = {
    "Uniform(0,1)": lambda n: np.random.uniform(0, 1, n),
    "Exponential(λ=2)": lambda n: np.random.exponential(0.5, n),
    "Binomial(10, 0.3)": lambda n: np.random.binomial(10, 0.3, n),
}

sample_sizes = [1, 2, 5, 30]
n_experiments = 10000

fig, axes = plt.subplots(len(distributions), len(sample_sizes), figsize=(16, 10))

for row, (name, dist_fn) in enumerate(distributions.items()):
    for col, n in enumerate(sample_sizes):
        means = [dist_fn(n).mean() for _ in range(n_experiments)]
        axes[row, col].hist(means, bins=50, density=True, alpha=0.7, color="steelblue")
        axes[row, col].set_title(f"{name}
n={n}", fontsize=9)
        if col == 0:
            axes[row, col].set_ylabel("Density")

plt.suptitle("Central Limit Theorem: Sample Means → Normal as n increases", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify CLT convergence
np.random.seed(42)
sample_sizes = [2, 5, 10, 30, 100, 500]
skewness_values = []
kurtosis_values = []

for n in sample_sizes:
    means = [np.random.exponential(1, n).mean() for _ in range(5000)]
    skewness_values.append(stats.skew(means))
    kurtosis_values.append(stats.kurtosis(means))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(sample_sizes, skewness_values, "o-")
ax1.axhline(0, color="r", linestyle="--", label="Normal (skew=0)")
ax1.set_xlabel("Sample size"); ax1.set_ylabel("Skewness"); ax1.legend()
ax2.plot(sample_sizes, kurtosis_values, "o-")
ax2.axhline(0, color="r", linestyle="--", label="Normal (kurtosis=0)")
ax2.set_xlabel("Sample size"); ax2.set_ylabel("Excess Kurtosis"); ax2.legend()
plt.suptitle("Convergence to Normal (Exponential source)")
plt.tight_layout()
plt.show()

## 3. Bayes Theorem

33935P(A|B) = rac{P(B|A) \cdot P(A)}{P(B)}33935

**Example:** Medical test with:
- Disease prevalence: 1%
- Test sensitivity (true positive rate): 95%
- Test specificity (true negative rate): 90%

If you test positive, what is the probability you actually have the disease?

In [ ]:
# Bayes Theorem - Medical Test Example
prevalence = 0.01  # P(Disease)
sensitivity = 0.95  # P(Positive | Disease)
specificity = 0.90  # P(Negative | No Disease)

false_positive_rate = 1 - specificity  # P(Positive | No Disease)

# P(Positive) = P(Pos|Disease)*P(Disease) + P(Pos|No Disease)*P(No Disease)
p_positive = sensitivity * prevalence + false_positive_rate * (1 - prevalence)

# P(Disease | Positive) = P(Pos|Disease) * P(Disease) / P(Positive)
p_disease_given_positive = (sensitivity * prevalence) / p_positive

print(f"P(Disease) = {prevalence:.2%}")
print(f"P(Positive) = {p_positive:.4f}")
print(f"P(Disease | Positive) = {p_disease_given_positive:.2%}")
print(f"
Surprising! Even with a positive test, only {p_disease_given_positive:.1%} chance of disease.")
print(f"This is because the disease is rare (base rate fallacy).")

In [ ]:
# Visualize how posterior changes with prevalence
prevalences = np.linspace(0.001, 0.5, 100)
posteriors = (sensitivity * prevalences) / (sensitivity * prevalences + false_positive_rate * (1 - prevalences))

plt.figure(figsize=(8, 5))
plt.plot(prevalences * 100, posteriors * 100, linewidth=2)
plt.axvline(1, color="r", linestyle="--", alpha=0.5, label="Prevalence=1%")
plt.axhline(p_disease_given_positive*100, color="g", linestyle="--", alpha=0.5, label=f"Posterior={p_disease_given_positive:.1%}")
plt.xlabel("Disease Prevalence (%)")
plt.ylabel("P(Disease | Positive Test) (%)")
plt.title("Bayes Theorem: How Prevalence Affects Posterior")
plt.legend()
plt.grid(True)
plt.show()

## 4. Hypothesis Testing

Hypothesis testing helps us make decisions from data:
- **H₀ (Null hypothesis):** No effect / no difference
- **H₁ (Alternative):** There IS an effect
- **p-value:** Probability of seeing data this extreme if H₀ is true
- **α (significance level):** Threshold for rejecting H₀ (typically 0.05)

In [ ]:
# Example: A/B test for a new feature
np.random.seed(42)

# Control group: mean engagement = 5.0 minutes
control = np.random.normal(5.0, 1.5, 200)
# Treatment group: mean engagement = 5.4 minutes (small improvement)
treatment = np.random.normal(5.4, 1.5, 200)

# Two-sample t-test
t_stat, p_value = stats.ttest_ind(control, treatment)

print("A/B Test Results:")
print(f"Control mean: {control.mean():.3f} min")
print(f"Treatment mean: {treatment.mean():.3f} min")
print(f"Difference: {treatment.mean() - control.mean():.3f} min")
print(f"
t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"
Conclusion at α=0.05: {"Reject H₀ (significant difference)" if p_value < 0.05 else "Fail to reject H₀"}")

In [ ]:
# Visualize the t-test
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Data distributions
ax1.hist(control, bins=30, alpha=0.5, label=f"Control (μ={control.mean():.2f})", density=True)
ax1.hist(treatment, bins=30, alpha=0.5, label=f"Treatment (μ={treatment.mean():.2f})", density=True)
ax1.set_title("Distribution of Groups")
ax1.legend()

# t-distribution with critical region
df = len(control) + len(treatment) - 2
x = np.linspace(-4, 4, 200)
ax2.plot(x, stats.t.pdf(x, df), "k-", linewidth=2)
ax2.axvline(t_stat, color="r", linewidth=2, label=f"t-stat={t_stat:.2f}")
crit = stats.t.ppf(0.025, df)
ax2.fill_between(x[x<=crit], stats.t.pdf(x[x<=crit], df), color="red", alpha=0.3, label="Rejection region")
ax2.fill_between(x[x>=-crit], stats.t.pdf(x[x>=-crit], df), color="red", alpha=0.1)
ax2.set_title(f"t-Distribution (df={df})")
ax2.legend()

plt.tight_layout()
plt.show()

## Key Takeaways

| Concept | ML Application |
|---------|---------------|
| Distributions | Modeling data, generative models |
| CLT | Why SGD works, confidence intervals |
| Bayes Theorem | Naive Bayes, Bayesian inference |
| Hypothesis Testing | A/B testing, feature selection |

**Remember:** Statistical thinking helps you avoid fooling yourself with data!